# Temas Tratados en el Trabajo Práctico 4

* Representación del Conocimiento y Razonamiento Lógico.

* Estrategias de resolución de hipótesis: Encadenamiento hacia Adelante, Encadenamiento hacia Atrás y Resolución por Contradicción.

* Representación basada en circuitos.

## Ejercicios Teóricos

1. ¿Qué es una inferencia?

2. ¿Cómo se verifica que un modelo se infiere de la base de conocimientos?

3. Observe la siguiente base de conocimiento:

$R1: b ∧ c → a$

$R2: d ∧ e → b$

$R3: g ∧ e → b$

$R4: e → c$

$R5: d$

$R6: e$

$R7: a ∧ g → f$

        3.1 ¿Cómo se puede probar que $a = True$ a través del encadenamiento hacia adelante? Este método solamente usa reglas ya incorporadas a la base de conocimiento para inferir la hipótesis, ¿qué propiedad debe tener el algoritmo para asegurar que esta inferencia sea posible?

        3.2 ¿Cómo se puede probar que $a = True$ a través del encadenamiento hacia atrás? Este método asigna un valor de verdad a la hipótesis y deriva las sentencias de la base de conocimiento, ¿qué propiedad debe tener el algoritmo para asegurar que esta derivación sea posible?

        3.3 Exprese la base de conocimiento en su Forma Normal Conjuntiva. A continuación, demuestre por contradicción que $a = True$.

4. Diseñe con lógica proposicional basada en circuitos las proposiciones *OrientadoDerecha* y *Agente ubicado en la casilla [1,2]* para el mundo de wumpus de 4x4. Dibuje el circuito correspondiente.

# Punto 4 - Wumpus 4x4: circuito de combinaciones orientacion x posicion (t -> t+1)
# Salidas buscadas: OrientadoDerecha^{t+1} y L[1,2]^{t+1}
#
# Idea: para producir L[1,2] en t+1, para CADA casilla origen (L[1,1], L[1,3], L[2,2])
# y CADA orientacion en t se arma un AND(orientacion_t, posicion_t). Su salida entra a una
# caja de GIRO (sin giro / girar 90 / girar -90 / girar 180) que deja al agente con la
# orientacion necesaria, y de ahi a la caja de TRASLACION (mover derecha/izquierda/abajo).
# Convencion de giro anclada al ejemplo pedido:  Abajo --(girar -90)--> Derecha.
# Rama "permanecer": si ya esta en L[1,2] y OrientadoDerecha, va directo a la 3a columna.
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle

fig, ax = plt.subplots(figsize=(17, 16))
ax.set_xlim(0, 18)
ax.set_ylim(-1.8, 24.5)
ax.axis("off")

pos, patch = {}, {}

def _add(key, x, y, w, h, face, edge, text, fs, lw=1.4, rounded=True, bold=False):
    if rounded:
        p = FancyBboxPatch((x - w/2, y - h/2), w, h,
                           boxstyle="round,pad=0.06", linewidth=lw,
                           edgecolor=edge, facecolor=face)
    else:
        p = Rectangle((x - w/2, y - h/2), w, h, linewidth=lw,
                      edgecolor=edge, facecolor=face)
    ax.add_patch(p)
    ax.text(x, y, text, ha="center", va="center", fontsize=fs,
            fontweight="bold" if bold else "normal")
    pos[key] = (x, y)
    patch[key] = p

def estado(key, x, y, text, kind):
    face, edge = ("#dbe6f3", "#1f77b4") if kind == "o" else ("#dff0d8", "#2ca02c")
    _add(key, x, y, 2.7, 0.9, face, edge, text, 8)

def gate(key, x, y, symbol):
    _add(key, x, y, 0.8, 0.8, "#ececec", "#333333", symbol, 11, rounded=False)

def accion(key, x, y, text, kind):
    face, edge = ("#ffe6c7", "#e8830c") if kind == "giro" else ("#e9d9f4", "#7d3ca3")
    _add(key, x, y, 2.0, 0.82, face, edge, text, 7.5, rounded=False)

def salida(key, x, y, text):
    _add(key, x, y, 2.9, 1.0, "#fff1ac", "#222222", text, 9, lw=2.4, bold=True)

def conectar(a, b, color="#444444", lw=1.3, alpha=1.0, ls="solid"):
    ax.add_patch(FancyArrowPatch(pos[a], pos[b], patchA=patch[a], patchB=patch[b],
                                 shrinkA=1, shrinkB=1, arrowstyle="-|>", mutation_scale=11,
                                 linewidth=lw, color=color, alpha=alpha, linestyle=ls,
                                 connectionstyle="arc3,rad=0", zorder=1))

X_ST, X_AND, X_GIRO, X_MOV, X_OR, X_OUT = 1.7, 5.2, 8.4, 11.4, 13.8, 16.2

# --- Col 0: estados en t ---
estado("O_Der", X_ST, 22.4, "OrientadoDerecha", "o")
estado("O_Arr", X_ST, 20.9, "OrientadoArriba", "o")
estado("O_Izq", X_ST, 19.4, "OrientadoIzquierda", "o")
estado("O_Abj", X_ST, 17.9, "OrientadoAbajo", "o")
estado("L11", X_ST, 13.6, "L[1,1]", "p")
estado("L12", X_ST, 11.0, "L[1,2]", "p")
estado("L13", X_ST, 8.0, "L[1,3]", "p")
estado("L22", X_ST, 4.0, "L[2,2]", "p")

# --- Col 1: compuertas AND (13) ---
gate("AND_perm", X_AND, 23.2, "&")
A_y = [21.2, 19.7, 18.2, 16.7]
for k, y in zip(["A_Der", "A_Arr", "A_Izq", "A_Abj"], A_y):
    gate(k, X_AND, y, "&")
B_y = [14.2, 12.7, 11.2, 9.7]
for k, y in zip(["B_Izq", "B_Arr", "B_Abj", "B_Der"], B_y):
    gate(k, X_AND, y, "&")
C_y = [7.0, 5.5, 4.0, 2.5]
for k, y in zip(["C_Abj", "C_Der", "C_Arr", "C_Izq"], C_y):
    gate(k, X_AND, y, "&")

# --- Col 2: cajas de giro (1:1 con los AND de A/B/C) ---
giro_of = {
    "A_Der": ("gA_id", "sin giro"),   "A_Arr": ("gA_90", "girar 90"),
    "A_Izq": ("gA_180", "girar 180"), "A_Abj": ("gA_m90", "girar -90"),
    "B_Izq": ("gB_id", "sin giro"),   "B_Arr": ("gB_m90", "girar -90"),
    "B_Abj": ("gB_90", "girar 90"),   "B_Der": ("gB_180", "girar 180"),
    "C_Abj": ("gC_id", "sin giro"),   "C_Der": ("gC_90", "girar 90"),
    "C_Arr": ("gC_180", "girar 180"), "C_Izq": ("gC_m90", "girar -90"),
}
for andk, (gk, lab) in giro_of.items():
    accion(gk, X_GIRO, pos[andk][1], lab, "giro")

# --- Col 3: cajas de traslacion ---
accion("mov_der", X_MOV, sum(A_y)/4, "mover derecha", "mov")
accion("mov_izq", X_MOV, sum(B_y)/4, "mover izquierda", "mov")
accion("mov_abj", X_MOV, sum(C_y)/4, "mover abajo", "mov")

# --- Col 4: compuertas OR ---
gate("OR_orient", X_OR, 20.0, ">=1")
gate("OR_pos", X_OR, 10.5, ">=1")

# --- Col 5: salidas en t+1 ---
salida("OUT_ODer", X_OUT, 20.0, "OrientadoDerecha")
salida("OUT_L12", X_OUT, 10.5, "L[1,2]")

# ---------------- aristas ----------------
CO = {"O_Der": "#1f77b4", "O_Arr": "#17a2b8", "O_Izq": "#8c564b", "O_Abj": "#d62728"}
orient_to_and = {
    "O_Der": ["A_Der", "B_Der", "C_Der", "AND_perm"],
    "O_Arr": ["A_Arr", "B_Arr", "C_Arr"],
    "O_Izq": ["A_Izq", "B_Izq", "C_Izq"],
    "O_Abj": ["A_Abj", "B_Abj", "C_Abj"],
}
for s, ands in orient_to_and.items():
    for a in ands:
        conectar(s, a, color=CO[s], lw=1.0, alpha=0.55)

pos_to_and = {
    "L11": ["A_Der", "A_Arr", "A_Izq", "A_Abj"],
    "L13": ["B_Izq", "B_Arr", "B_Abj", "B_Der"],
    "L22": ["C_Abj", "C_Der", "C_Arr", "C_Izq"],
    "L12": ["AND_perm"],
}
for s, ands in pos_to_and.items():
    for a in ands:
        conectar(s, a, color="#2ca02c", lw=1.0, alpha=0.55)

for andk, (gk, _) in giro_of.items():          # AND -> caja de giro
    conectar(andk, gk, color="#444444", lw=1.3)

for gk, _ in giro_of.values():                  # caja de giro -> caja de mover
    dest = {"A": "mov_der", "B": "mov_izq", "C": "mov_abj"}[gk[1]]
    conectar(gk, dest, color="#7d3ca3", lw=1.2)

for gk in ["gA_id", "gA_90", "gA_180", "gA_m90"]:   # grupo A (giros) -> OR_orient
    conectar(gk, "OR_orient", color="#e8830c", lw=1.2)
conectar("AND_perm", "OR_orient", color="#666666", lw=1.1, ls="dashed", alpha=0.8)

for mk in ["mov_der", "mov_izq", "mov_abj"]:        # movs -> OR_pos
    conectar(mk, "OR_pos", color="#7d3ca3", lw=1.3)
conectar("AND_perm", "OR_pos", color="#666666", lw=1.1, ls="dashed", alpha=0.8)

conectar("OR_orient", "OUT_ODer", color="#222222", lw=1.6)
conectar("OR_pos", "OUT_L12", color="#222222", lw=1.6)

# ---------------- rotulos / leyenda ----------------
ax.text(X_ST, 23.9, "t", ha="center", fontsize=15, fontweight="bold")
ax.text(X_OUT, 23.9, "t + 1", ha="center", fontsize=15, fontweight="bold")
ax.text(X_AND, 22.0, "grupo A  .  origen L[1,1]  .  mover derecha", ha="left", fontsize=8, style="italic", color="#555")
ax.text(X_AND, 15.0, "grupo B  .  origen L[1,3]  .  mover izquierda", ha="left", fontsize=8, style="italic", color="#555")
ax.text(X_AND, 7.8, "grupo C  .  origen L[2,2]  .  mover abajo", ha="left", fontsize=8, style="italic", color="#555")

leg = [("orientacion en t", "#dbe6f3", "#1f77b4"),
       ("posicion en t", "#dff0d8", "#2ca02c"),
       ("accion: giro", "#ffe6c7", "#e8830c"),
       ("accion: traslacion", "#e9d9f4", "#7d3ca3"),
       ("compuerta AND (&) / OR (>=1)", "#ececec", "#333333")]
for i, (txt, fc, ec) in enumerate(leg):
    ax.add_patch(Rectangle((0.5 + i*3.5, 0.3), 0.5, 0.5, facecolor=fc, edgecolor=ec, linewidth=1.3))
    ax.text(1.1 + i*3.5, 0.55, txt, ha="left", va="center", fontsize=7.3)

ax.set_title("Punto 4 - Wumpus 4x4 : combinaciones orientacion x posicion  (t -> t+1)\n"
             "salidas: OrientadoDerecha  y  L[1,2]", fontsize=12)
ax.text(9, -1.2,
        "Nota: el update de orientacion es en rigor independiente de la posicion; "
        "el diagrama muestra las combinaciones relevantes para producir L[1,2] en t+1.",
        ha="center", va="center", fontsize=7, style="italic", color="#666")

plt.tight_layout()
plt.show()


5. El nonograma es un juego en el cual se posee un tablero en blanco y cada fila y columna presenta información sobre la longitud de un bloque en dicha fila/columna. Además, la leyenda puede indicar más de un número, indicando esto que existen varios bloques de las longitudes mostradas por la leyenda y en el mismo orden, separados por al menos un espacio vacío.
Resuelva el nonograma de la imagen de abajo escribiendo en primer lugar cada regla que puede incorporarse a la base de conocimientos inicial e incorporando cada inferencia que realice

NONOGRAMA 4x4 - Punto 5

Notacion: Crc = casilla de la fila r y columna c.  ∧ = AND , ∨ = OR.
Verdadero = casilla llena.  Falso = casilla vacia.
Leyendas filas:    R1="1 1"  R2="4"  R3="2 1"  R4="3"
Leyendas columnas: C1="3"    C2="3"  C3="2 1"  C4="3"

=== REGLAS DE FILAS (base de conocimiento inicial) ===
R1 (fila 1, "1 1"):  (C11 ∧ C13)  ∨  (C11 ∧ C14)  ∨  (C12 ∧ C14)
R2 (fila 2, "4"):    C21 ∧ C22 ∧ C23 ∧ C24
R3 (fila 3, "2 1"):  C31 ∧ C32 ∧ (NO C33) ∧ C34        [unica ubicacion en 4 celdas]
R4 (fila 4, "3"):    (C41 ∧ C42 ∧ C43)  ∨  (C42 ∧ C43 ∧ C44)

=== REGLAS DE COLUMNAS ===
C1 (col 1, "3"):    (C11 ∧ C21 ∧ C31)  ∨  (C21 ∧ C31 ∧ C41)
C2 (col 2, "3"):    (C12 ∧ C22 ∧ C32)  ∨  (C22 ∧ C32 ∧ C42)
C3 (col 3, "2 1"):  C13 ∧ C23 ∧ (NO C33) ∧ C43         [unica ubicacion en 4 celdas]
C4 (col 4, "3"):    (C14 ∧ C24 ∧ C34)  ∨  (C24 ∧ C34 ∧ C44)

=== INFERENCIAS ===
I1  (de R2): C21, C22, C23, C24 = verdaderos  (fila 2 completa).

I2  (de R3): "2 1" en 4 celdas tiene una sola ubicacion posible
     (bloque de 2 en col 1-2, hueco en col 3, bloque de 1 en col 4):
     C31, C32, C34 = verdaderos ;  C33 = falso.

I3  (de C3): "2 1" en 4 celdas, una sola ubicacion posible
     (bloque de 2 en fila 1-2, hueco en fila 3, bloque de 1 en fila 4):
     C13, C23, C43 = verdaderos ;  C33 = falso  (coincide con I2).

I4  (de R1 + I3): como C13 es verdadero, la unica opcion de R1 compatible es (C11 ∧ C13):
     C11 = verdadero ;  C12 = falso ;  C14 = falso.

I5  (de C1 + I4): como C11 es verdadero, la unica opcion de C1 compatible es (C11 ∧ C21 ∧ C31):
     C31 = verdadero (coincide con I2) ;  C41 = falso.

I6  (de R4 + I5): como C41 es falso, la unica opcion de R4 compatible es (C42 ∧ C43 ∧ C44):
     C42, C43, C44 = verdaderos.

I7  (de C2 + I4): como C12 es falso, la unica opcion de C2 compatible es (C22 ∧ C32 ∧ C42):
     C22, C32, C42 = verdaderos  (consistente con lo ya inferido).

I8  (de C4 + I4): como C14 es falso, la unica opcion de C4 compatible es (C24 ∧ C34 ∧ C44):
     C24, C34, C44 = verdaderos  (consistente con lo ya inferido).

=== SOLUCION FINAL ===
Llenas (verdadero): C11, C13, C21, C22, C23, C24, C31, C32, C34, C42, C43, C44
Vacias (falso):     C12, C14, C33, C41

        c1 c2 c3 c4     leyenda fila
 f1  [  X  .  X  .  ]     1 1
 f2  [  X  X  X  X  ]     4
 f3  [  X  X  .  X  ]     2 1
 f4  [  .  X  X  X  ]     3
 col    3  3  2.1 3


## Ejercicios de Implementación

> Recuerde adjuntar en la presentación el prompt inicial que ha utilizado para cada ejercicio de implementación y si considera que le dio la información completa para resolver el ejercicio o qué cambios adicionales tuvo que pedir de manera iterativa.

6. Implementar un motor de inferencia con encadenamiento hacia adelante. Pruébelo con las proposiciones del ejercicio 3.

7. Implementar un motor de inferencia con encadenamiento hacia atrás. Pruébelo con las proposiciones del ejercicio 3.

8. Implementar un motor de inferencia por contradicción que detecte si el conjunto de proposiciones del ejercicio 3 es inconsistente.

# Bibliografía

[Russell, S. & Norvig, P. (2004) _Inteligencia Artificial: Un Enfoque Moderno_. Pearson Educación S.A. (2a Ed.) Madrid, España](https://www.academia.edu/8241613/Inteligencia_Aritificial_Un_Enfoque_Moderno_2da_Edici%C3%B3n_Stuart_J_Russell_y_Peter_Norvig)

[Poole, D. & Mackworth, A. (2023) _Artificial Intelligence: Foundations of Computational Agents_. Cambridge University Press (3a Ed.) Vancouver, Canada](https://artint.info/3e/html/ArtInt3e.html)